# Megatron-LM 有哪些并行配置？

Megatron-LM 提供了丰富的并行配置选项，支持从单机多卡到大规模集群的灵活部署。以下是完整的并行配置体系：

| 并行方式           | 切分维度 | 每个 GPU 持有 | 通信内容                   | 适用场景        |
| -------------- | ---- | --------- | ---------------------- | ----------- |
| **数据并行 (DP)**  | 批次维度 | 完整模型副本    | 梯度 (all-reduce)        | 模型小、batch 大 |
| **张量并行 (TP)**  | 层内维度 | 每层的一部分    | 激活值 (all-reduce)       | 单层太大、单卡放不下  |
| **流水线并行 (PP)** | 层间维度 | 部分层完整     | 层间 hidden states (p2p) | 模型层数多、超长序列  |
| **序列并行 (SP)**  | 序列维度 | 部分 token  | 激活值 (all-gather)       | 超长上下文       |


## 并行维度总览

In [ ]:
Megatron-LM 并行维度（5D 并行）：

    Data Parallel (DP) ────────→ 数据分片到不同 GPU
         ↑
    Tensor Parallel (TP) ──────→ 层内参数分片（Attention/MLP）
         ↑
    Pipeline Parallel (PP) ────→ 层间分配到不同 GPU
         ↑
    Sequence Parallel (SP) ────→ 序列维度分片（与 TP 配合）
         ↑
    Context Parallel (CP) ─────→ 长上下文分片（Ring Attention）
         ↑
    Expert Parallel (EP) ──────→ MoE 专家分片（MoE 模型专用）

## 各并行维度详解

### <font color='red'>张量并行 (Tensor Parallel, TP)</font>
#### 作用：将单层内的参数（如 Attention Q/K/V 投影、MLP 的线性层）分片到多个 GPU。
- 张量并行（TP）<font color='red'>拆分的是 Transformer 层内的线性投影矩阵（QKV、Output、MLP Up/Down）</font>，按矩阵的列或行维度切分到不同 GPU，而不是拆分整个 Transformer 层或整个 Attention 模块。

In [ ]:
TP=2 时的 Attention 层：

GPU 0: [Q0|K0|V0] ──→ 计算局部 attention ──→ [O0]
GPU 1: [Q1|K1|V1] ──→ 计算局部 attention ──→ [O1]
         ↑                                    ↓
         └──────── All-Gather O ──────────────┘
                    合并输出

#### TP 拆分的对象


In [ ]:
┌─────────────────────────────────────────────────────────────┐
│              Transformer 层的内部结构                          │
├─────────────────────────────────────────────────────────────┤
│                                                              │
│  Input: hidden_states [batch, seq, hidden]                   │
│         ↓                                                    │
│  ┌─────────────────────────────────────────────────────┐      │
│  │              LayerNorm (SP 拆分)                     │      │
│  └─────────────────────────────────────────────────────┘      │
│         ↓                                                    │
│  ┌─────────────────────────────────────────────────────┐      │
│  │              Self-Attention                         │      │
│  │  ┌─────────────────────────────────────────────┐    │      │
│  │  │  QKV Projection: [hidden, 3×hidden]          │◄───┼── TP │
│  │  │      ColumnParallelLinear                    │    │      │
│  │  │      TP=2: [hidden, 1.5×hidden] per GPU      │    │      │
│  │  └─────────────────────────────────────────────┘    │      │
│  │         ↓                                            │      │
│  │  ┌─────────────────────────────────────────────┐    │      │
│  │  │  Attention Compute (Q×K^T, Softmax, ×V)     │    │      │
│  │  │      每个 GPU 计算局部 attention              │    │      │
│  │  │      需要 All-Gather 或序列并行通信         │    │      │
│  │  └─────────────────────────────────────────────┘    │      │
│  │         ↓                                            │      │
│  │  ┌─────────────────────────────────────────────┐    │      │
│  │  │  Output Projection: [hidden, hidden]          │◄───┼── TP │
│  │  │      RowParallelLinear                       │    │      │
│  │  │      TP=2: [0.5×hidden, hidden] per GPU     │    │      │
│  │  └─────────────────────────────────────────────┘    │      │
│  └─────────────────────────────────────────────────────┘      │
│         ↓                                                    │
│  ┌─────────────────────────────────────────────────────┐      │
│  │              LayerNorm (SP 拆分)                     │      │
│  └─────────────────────────────────────────────────────┘      │
│         ↓                                                    │
│  ┌─────────────────────────────────────────────────────┐      │
│  │              MLP (Feed-Forward)                     │      │
│  │  ┌─────────────────────────────────────────────┐    │      │
│  │  │  Gate/Up Projection: [hidden, 2×ffn_hidden] │◄───┼── TP │
│  │  │      ColumnParallelLinear                    │    │      │
│  │  └─────────────────────────────────────────────┘    │      │
│  │         ↓                                            │      │
│  │  ┌─────────────────────────────────────────────┐    │      │
│  │  │  Down Projection: [ffn_hidden, hidden]      │◄───┼── TP │
│  │  │      RowParallelLinear                       │    │      │
│  │  └─────────────────────────────────────────────┘    │      │
│  └─────────────────────────────────────────────────────┘      │
│                                                              │
└─────────────────────────────────────────────────────────────┘

#### TP 具体拆分哪些层？
##### <font color='red'>被 TP 拆分的层（Column/Row Parallel）</font>

| 层                    | 原始形状                     | TP=2 后每 GPU                | 类型             |
| -------------------- | ------------------------ | -------------------------- | -------------- |
| **QKV Projection**   | `[hidden, 3×hidden]`     | `[hidden, 1.5×hidden]`     | <font color='red'>ColumnParallel</font> |
| **Attention Output** | `[hidden, hidden]`       | `[0.5×hidden, hidden]`     | <font color='red'>RowParallel </font>   |
| **MLP Gate/Up**      | `[hidden, 2×ffn_hidden]` | `[hidden, ffn_hidden]`     | <font color='red'>ColumnParallel</font> |
| **MLP Down**         | `[ffn_hidden, hidden]`   | `[0.5×ffn_hidden, hidden]` | <font color='red'>RowParallel </font>   |


##### <font color='red'>不被 TP 拆分的层</font>

| 层                                 | 原因                         |
| --------------------------------- | -------------------------- |
| **LayerNorm**                     | 参数少，用 Sequence Parallel 替代 |
| **Embedding**                     | 通常用其他并行方式，或复制              |
| **Attention 计算** (Q×K^T, Softmax) | 是计算操作，不是参数矩阵               |
| **Bias**                          | 通常与对应线性层一起处理               |

#### ColumnParallel vs RowParallel
#####  ColumnParallelLinear（列并行）
- 特点：输入相同，输出分片，按输出维度切分。

In [ ]:
输入: [batch, seq, hidden] ──→ 所有 GPU 相同

GPU 0: W0 [hidden, hidden/TP]  ──→ 输出 Y0 [batch, seq, hidden/TP]
GPU 1: W1 [hidden, hidden/TP]  ──→ 输出 Y1 [batch, seq, hidden/TP]

合并: [Y0 | Y1] = Y [batch, seq, hidden]  (All-Gather 或 concat)

##### RowParallelLinear（行并行）
- 特点：输入分片，输出相同，按输入维度切分。

In [ ]:
输入: [batch, seq, hidden] ──→ 先分片

GPU 0: X0 [batch, seq, hidden/TP]  ──→ W0 [hidden/TP, hidden]  ──→ Y0
GPU 1: X1 [batch, seq, hidden/TP]  ──→ W1 [hidden/TP, hidden]  ──→ Y1

合并: Y0 + Y1 = Y [batch, seq, hidden]  (All-Reduce 求和)

#### 代码示例：Megatron 中的 TP 实现

In [ ]:
# Megatron 核心：张量并行的线性层实现

from megatron.core.tensor_parallel.layers import ColumnParallelLinear, RowParallelLinear

class ParallelAttention(nn.Module):
    def __init__(self, hidden_size, num_attention_heads, tp_size=1):
        super().__init__()
        
        # QKV 投影：列并行（输出维度拆分）
        self.query_key_value = ColumnParallelLinear(
            hidden_size,
            3 * hidden_size,           # Q + K + V
            gather_output=False,        # 不 All-Gather，保持分片
            init_method=init_method,
            bias=True,
            async_tensor_model_parallel_allreduce=True,  # 异步通信
        )
        
        # Attention 输出投影：行并行（输入维度拆分）
        self.dense = RowParallelLinear(
            hidden_size,
            hidden_size,
            input_is_parallel=True,     # 输入已经是分片的
            init_method=init_method,
            bias=True,
            skip_bias_add=True,
        )
    
    def forward(self, hidden_states, attention_mask):
        # hidden_states: [batch, seq, hidden]
        
        # 1. QKV 投影（列并行）
        # 各 GPU 输出: [batch, seq, 1.5×hidden] (TP=2)
        query_key_value = self.query_key_value(hidden_states)
        
        # 2. 切分 Q, K, V
        # 各 GPU 有完整的 Q, K, V 分片
        query, key, value = query_key_value.chunk(3, dim=-1)
        
        # 3. Attention 计算（各 GPU 计算局部）
        # 需要 All-Gather 或序列并行通信
        attention_output = self._flash_attention(query, key, value, attention_mask)
        
        # 4. 输出投影（行并行）
        # 输入已经是分片的，输出 All-Reduce 求和
        output = self.dense(attention_output)
        
        return output

In [ ]:
class ParallelMLP(nn.Module):
    def __init__(self, hidden_size, ffn_hidden_size, tp_size=1):
        super().__init__()
        
        # Gate/Up 投影：列并行
        self.linear_fc1 = ColumnParallelLinear(
            hidden_size,
            2 * ffn_hidden_size,      # Gate + Up (Gated MLP)
            gather_output=False,
            init_method=init_method,
            bias=False,
        )
        
        # Down 投影：行并行
        self.linear_fc2 = RowParallelLinear(
            ffn_hidden_size,
            hidden_size,
            input_is_parallel=True,
            init_method=init_method,
            bias=False,
        )
    
    def forward(self, hidden_states):
        # 1. Gate/Up 投影（列并行）
        # 输出分片: [batch, seq, ffn_hidden] per GPU
        gate_up = self.linear_fc1(hidden_states)
        gate, up = gate_up.chunk(2, dim=-1)
        
        # 2. SwiGLU 激活
        intermediate = F.silu(gate) * up
        
        # 3. Down 投影（行并行）
        # 输入分片，输出 All-Reduce
        output = self.linear_fc2(intermediate)
        
        return output

#### 先经过列并行，然后再经过行并行？
- 这是一个关于张量并行（TP）设计的核心问题。先经过 ColumnParallel（列并行）再经过 RowParallel（行并行）的组合，是为了消除中间结果的 All-Gather 通信，将两次通信合并为一次，从而大幅提升效率。

##### 直观对比：两种组合方式
- 方式 A：Column → Row（Megatron 采用，最优）

In [ ]:
输入 X [batch, seq, hidden] ──→ 所有 GPU 相同

┌─────────────────────────────────────────────────────────────┐
│  GPU 0                        GPU 1                          │
│                                                              │
│  ColumnParallel(QKV)          ColumnParallel(QKV)           │
│  W0: [H, 1.5H]                W1: [H, 1.5H]                │
│  ↓                            ↓                             │
│  Y0 = X @ W0                  Y1 = X @ W1                 │
│  [B,S,1.5H]                   [B,S,1.5H]                  │
│                                                              │
│  中间结果已经是分片的！                                       │
│  不需要 All-Gather！                                          │
│                                                              │
│  RowParallel(Output)          RowParallel(Output)           │
│  V0: [0.5H, H]                V1: [0.5H, H]                │
│  ↓                            ↓                             │
│  Z0 = Y0[:,:,:0.5H] @ V0      Z1 = Y0[:,:,0.5H:] @ V1     │
│  [B,S,H]                      [B,S,H]                      │
│                                                              │
│  All-Reduce: Z = Z0 + Z1      ← 只需要一次通信！            │
└─────────────────────────────────────────────────────────────┘

- 方式 B：Row → Column（不推荐）

In [ ]:
输入 X [batch, seq, hidden] ──→ 先分片

┌─────────────────────────────────────────────────────────────┐
│  GPU 0                        GPU 1                          │
│                                                              │
│  RowParallel(QKV)             RowParallel(QKV)              │
│  W0: [0.5H, 3H]               W1: [0.5H, 3H]              │
│  ↓                            ↓                             │
│  Y0 = X[:,:,:0.5H] @ W0       Y1 = X[:,:,0.5H:] @ W1       │
│  [B,S,3H]                     [B,S,3H]                    │
│                                                              │
│  ⚠️ 需要 All-Gather！                                        │
│  Y = [Y0 | Y1] = [B,S,6H]    ← 中间结果要合并！             │
│                                                              │
│  ColumnParallel(Output)       ColumnParallel(Output)        │
│  V0: [6H, 0.5H]               V1: [6H, 0.5H]              │
│  ↓                            ↓                             │
│  Z0 = Y @ V0                  Z1 = Y @ V1                 │
│                                                              │
│  All-Gather: Z = [Z0 | Z1]   ← 又需要一次通信！            │
└─────────────────────────────────────────────────────────────┘

##### 核心原理：通信次数对比

| 组合方式             | 中间通信             | 最终通信             | 总通信次数      |
| ---------------- | ---------------- | ---------------- | ---------- |
| **Column → Row** | ❌ 无              | ✅ 1 次 All-Reduce | **1 次**    |
| Row → Column     | ✅ 1 次 All-Gather | ✅ 1 次 All-Gather | **2 次**    |
| Column → Column  | ❌ 无              | ✅ 1 次 All-Gather | 1 次（但输出分片） |
| Row → Row        | ✅ 1 次 All-Reduce | ✅ 1 次 All-Reduce | 2 次        |

##### 关键洞察：
- ColumnParallel 的输出是分片的（按列切分）
- RowParallel 的输入是分片的（按行切分）
- 分片的输出直接作为分片的输入 → 完美衔接，无需通信！

##### 数学证明
-  ColumnParallel 的输出 = RowParallel 的输入

In [ ]:
ColumnParallelLinear:
  输入: X [B, S, H]           (所有 GPU 相同)
  权重: W = [W0 | W1]         (按列切分)
  输出: Y = X @ W = [X@W0 | X@W1] = [Y0 | Y1]
  
  GPU 0 持有 Y0 = X @ W0      [B, S, H/TP]
  GPU 1 持有 Y1 = X @ W1      [B, S, H/TP]

RowParallelLinear:
  输入: Y = [Y0 | Y1]         (按行切分，即按列切分的输出)
  权重: V = [V0; V1]          (按行切分)
  
  GPU 0 计算: Z0 = Y0 @ V0    [B, S, H]
  GPU 1 计算: Z1 = Y1 @ V1    [B, S, H]
  
  最终: Z = Z0 + Z1           (All-Reduce 求和)

- 为什么能直接衔接？

In [ ]:
Y = X @ W = X @ [W0 | W1] = [X@W0 | X@W1] = [Y0 | Y1]

Z = Y @ V = [Y0 | Y1] @ [V0; V1] = Y0@V0 + Y1@V1 = Z0 + Z1

关键：矩阵乘法的分块性质
  [A | B] @ [C; D] = A@C + B@D

#### 完整 Attention 层的通信流程

In [ ]:
┌─────────────────────────────────────────────────────────────┐
│              Attention 层（TP=2，Column→Row）                 │
├─────────────────────────────────────────────────────────────┤
│                                                              │
│  输入 X: [B, S, H]                                           │
│  所有 GPU 都有完整的 X（广播或复制）                           │
│                                                              │
│  ┌─────────────────────────────────────────────────────┐      │
│  │  ColumnParallelLinear(QKV)                         │      │
│  │  GPU 0: W_qkv_0 [H, 1.5H]                          │      │
│  │  GPU 1: W_qkv_1 [H, 1.5H]                          │      │
│  │                                                     │      │
│  │  Y0 = X @ W_qkv_0   [B, S, 1.5H]                   │      │
│  │  Y1 = X @ W_qkv_1   [B, S, 1.5H]                   │      │
│  │                                                     │      │
│  │  ✓ 输出天然分片，无需通信                            │      │
│  └─────────────────────────────────────────────────────┘      │
│                         ↓                                    │
│  各 GPU 切分 Q, K, V（本地操作，无通信）                      │
│                         ↓                                    │
│  ┌─────────────────────────────────────────────────────┐      │
│  │  Flash Attention（各 GPU 计算局部）                   │      │
│  │  注意：可能需要序列并行通信（SP）或 All-Gather        │      │
│  │  但这不是 TP 的问题，是 Attention 算法本身           │      │
│  └─────────────────────────────────────────────────────┘      │
│                         ↓                                    │
│  attention_output: [B, S, H]（各 GPU 有完整或分片结果）      │
│                         ↓                                    │
│  ┌─────────────────────────────────────────────────────┐      │
│  │  RowParallelLinear(Output)                          │      │
│  │  GPU 0: V_0 [0.5H, H]                               │      │
│  │  GPU 1: V_1 [0.5H, H]                               │      │
│  │                                                     │      │
│  │  输入 attention_output 需要按列分片                  │      │
│  │  如果 attention 输出是完整的，需要先做 split          │      │
│  │                                                     │      │
│  │  Z0 = attn_out_0 @ V_0  [B, S, H]                  │      │
│  │  Z1 = attn_out_1 @ V_1  [B, S, H]                  │      │
│  │                                                     │      │
│  │  Z = All-Reduce(Z0 + Z1)  ← 唯一一次通信！          │      │
│  └─────────────────────────────────────────────────────┘      │
│                         ↓                                    │
│  输出: [B, S, H] 所有 GPU 相同（All-Reduce 后）              │
│                                                              │
└─────────────────────────────────────────────────────────────┘

#### MLP 层同样适用

In [ ]:
MLP (Gated MLP/SwiGLU):

┌─────────────────────────────────────────────────────────────┐
│  ColumnParallelLinear(Gate/Up)                               │
│  GPU 0: W_gu_0 [H, FFN]     GPU 1: W_gu_1 [H, FFN]         │
│  ↓                                                           │
│  Y0 = X @ W_gu_0            Y1 = X @ W_gu_1                │
│  [B, S, FFN]                [B, S, FFN]                    │
│                                                              │
│  ✓ 输出分片，直接衔接 RowParallel                            │
│                                                              │
│  RowParallelLinear(Down)                                     │
│  GPU 0: V_0 [0.5FFN, H]     GPU 1: V_1 [0.5FFN, H]        │
│  ↓                                                           │
│  Z0 = Y0[:,:,:0.5FFN] @ V_0                                 │
│  Z1 = Y0[:,:,0.5FFN:] @ V_1                                 │
│                                                              │
│  Z = All-Reduce(Z0 + Z1)  ← 唯一一次通信                   │
└─────────────────────────────────────────────────────────────┘

#### 配置参数：

In [ ]:
--tensor-model-parallel-size 4    # TP=4

#### 适用场景：
- 单卡放不下某一层（如 hidden_size=12288 的线性层）
- 通常 TP ≤ 8（受 NVLink 带宽限制，同节点内）

### <font color='red'>流水线并行 (Pipeline Parallel, PP)</font>
#### 作用：将模型的不同层分配到不同 GPU，形成流水线。

In [ ]:
PP=4 时的 24 层模型：

GPU 0 (Stage 0): Layer 0-5
GPU 1 (Stage 1): Layer 6-11
GPU 2 (Stage 2): Layer 12-17
GPU 3 (Stage 3): Layer 18-23

数据流：
  micro-batch 0: [F0]→[F1]→[F2]→[F3]→[B3]→[B2]→[B1]→[B0]
  micro-batch 1:     [F0]→[F1]→[F2]→[F3]→[B3]→[B2]→[B1]→[B0]
  micro-batch 2:         [F0]→[F1]→[F2]→[F3]→[B3]→[B2]→[B1]→[B0]

#### 配置参数：

In [ ]:
--pipeline-model-parallel-size 4    # PP=4
--num-layers-per-virtual-pipeline-stage 2  # 虚拟流水线（可选）

#### 调度策略：

| 策略               | 参数                                | 特点                |
| ---------------- | --------------------------------- | ----------------- |
| GPipe            | `--pipeline-schedule gpipe`       | 填充后全部前向，再全部反向，显存高 |
| 1F1B             | `--pipeline-schedule 1f1b`        | 来一个前向一个反向，显存低，推荐  |
| Interleaved 1F1B | `--pipeline-schedule interleaved` | 每层拆多个 chunk，气泡更小  |


### <font color='red'>数据并行 (Data Parallel, DP)</font>
#### 作用：将数据分片到不同 GPU，各 GPU 持有完整模型副本（或分片）。

In [ ]:
DP=4 时的数据分片：

Batch 1024 → 拆分为 4 个 micro-batch (256 each)

GPU 0: micro-batch 0 → 前向 → 反向 → 梯度
GPU 1: micro-batch 1 → 前向 → 反向 → 梯度
GPU 2: micro-batch 2 → 前向 → 反向 → 梯度
GPU 3: micro-batch 3 → 前向 → 反向 → 梯度
         ↓
    All-Reduce 同步梯度
         ↓
    各 GPU 更新相同参数

#### 配置参数：

In [ ]:
--data-parallel-size 4              # DP=4（通常自动计算）
--micro-batch-size 2                # 每 GPU 的 batch
--global-batch-size 64              # 总 batch = DP × micro_batch × grad_accum

#### 与 ZeRO/FSDP 结合：

In [ ]:
--use-distributed-optimizer         # 启用分布式优化器

--data-parallel-sharding-strategy optim          # ZeRO-1
--data-parallel-sharding-strategy optim_grads    # ZeRO-2
--data-parallel-sharding-strategy optim_grads_params  # ZeRO-3/FSDP

### <font color='red'>序列并行 (Sequence Parallel, SP)</font>
#### 作用：将输入序列维度分片，<font color='blue'>与 TP 配合使用，减少激活值显存。</font>

In [ ]:
SP=2 时的 LayerNorm/Dropout：

输入: [seq_len=4096, hidden=4096]

GPU 0: [0:2048, :]     # 前一半序列
GPU 1: [2048:4096, :]  # 后一半序列



##### 注意：<font color='red'>Attention 仍用 TP，但 LayerNorm/Dropout 用 SP</font>

#### 配置参数：


In [ ]:
--sequence-parallel            # 启用序列并行
--tensor-model-parallel-size 4  # SP 自动等于 TP

#### 原理：<font color='red'>TP 拆分的层（如 Linear）需要 All-Gather 输出，但 LayerNorm/Dropout 不需要，所以可以在序列维度分片。</font>

### <font color='red'>上下文并行 (Context Parallel, CP)</font>
#### 作用：长序列训练时，将序列分片到多个 GPU，使用 Ring Attention 减少显存。

In [ ]:
CP=4 时的长序列处理：

序列长度 1M tokens → 拆分为 4 段 (256K each)

GPU 0: tokens [0:256K]      ──┐
GPU 1: tokens [256K:512K]   ──┼── Ring Attention
GPU 2: tokens [512K:768K]   ──┤    各 GPU 只计算局部 attention
GPU 3: tokens [768K:1M]     ──┘    通过 P2P 传递 KV cache

通信模式：环形传递，每步只与相邻 GPU 通信

#### 配置参数：

In [ ]:
--context-parallel-size 4       # CP=4
--context-parallel-algorithm ring  # Ring Attention
--seq-length 1048576            # 1M 长序列

### <font color='red'>专家并行 (Expert Parallel, EP) — MoE 专用</font>
#### 作用：Mixture-of-Experts (MoE) 模型中，将不同专家分配到不同 GPU。

In [ ]:
EP=4 时的 MoE 层：

输入: [batch, hidden] → Router 选择 top-2 专家

GPU 0: Expert 0, Expert 1
GPU 1: Expert 2, Expert 3
GPU 2: Expert 4, Expert 5
GPU 3: Expert 6, Expert 7

All-to-All 通信：将 token 路由到对应专家的 GPU

#### 配置参数：

In [ ]:
--num-experts 8                 # 8 个专家
--expert-model-parallel-size 4   # EP=4
--moe-router-load-balancing-type aux_loss  # 负载均衡
--moe-token-dispatcher-type alltoall       # All-to-All 通信

## 并行维度组合关系
###  维度关系公式

In [ ]:
总 GPU 数 = TP × PP × DP × CP × EP

示例配置：
  TP=4, PP=4, DP=4, CP=1, EP=1
  总 GPU = 4 × 4 × 4 = 64

  8 节点 × 8 GPU = 64 GPU ✓

### 各维度的硬件亲和性

| 并行维度   | 最佳硬件拓扑          | 通信方式                      | 带宽要求 |
| ------ | --------------- | ------------------------- | ---- |
| **TP** | <font color='blue'>同节点内（NVLink）   </font> | <font color='red'>All-Reduce/All-Gather</font>     | 极高   |
| **SP** | <font color='blue'>同节点内（NVLink）   </font> | <font color='red'>All-Reduce</font>                | 极高   |
| **PP** | 跨节点（InfiniBand） | <font color='red'>P2P 发送/接收  </font>               | 中    |
| **CP** | 跨节点（InfiniBand） | <font color='red'>P2P Ring</font>                  | 中    |
| **DP** | 跨节点（InfiniBand） | <font color='red'>All-Reduce/Reduce-Scatter</font> | 高    |
| **EP** | 跨节点（InfiniBand） | <font color='red'>All-to-All </font>               | 高    |


## 完整配置示例
### 70B 模型训练配置

In [ ]:
#!/bin/bash
# 70B 模型，64 GPU（8 节点 × 8 GPU）

torchrun \
    --nproc_per_node=8 \
    --nnodes=8 \
    --node_rank=$NODE_RANK \
    --master_addr=$MASTER_ADDR \
    --master_port=29500 \
    pretrain_gpt.py \
    \
    # ===== 模型配置 =====
    --num-layers 80 \
    --hidden-size 8192 \
    --num-attention-heads 64 \
    --ffn-hidden-size 28672 \
    --seq-length 4096 \
    --max-position-embeddings 4096 \
    --vocab-size 32000 \
    \
    # ===== 并行配置 =====
    --tensor-model-parallel-size 4 \        # TP=4（同节点 NVLink）
    --pipeline-model-parallel-size 4 \      # PP=4（跨节点）
    --context-parallel-size 1 \            # CP=1（短序列不需要）
    --sequence-parallel \                   # 启用 SP（=TP）
    --use-distributed-optimizer \            # ZeRO-1 优化器
    --data-parallel-sharding-strategy optim \ # 仅拆分优化器状态
    \
    # ===== 训练配置 =====
    --micro-batch-size 1 \
    --global-batch-size 1024 \
    --train-iters 500000 \
    --lr 1.0e-4 \
    --min-lr 1.0e-5 \
    --lr-decay-style cosine \
    --lr-warmup-iters 2000 \
    --weight-decay 0.1 \
    --clip-grad 1.0 \
    --bf16 \
    \
    # ===== 性能优化 =====
    --overlap-grad-reduce \        # 重叠梯度通信
    --overlap-param-gather \       # 重叠参数收集
    --recompute-activations \      # 激活值重计算
    --recompute-granularity full \ # 全层重计算
    \
    # ===== 数据 =====
    --data-path /path/to/preprocessed_data \
    --split 949,50,1 \
    \
    # ===== 检查点 =====
    --save /path/to/checkpoints \
    --save-interval 1000 \
    --load /path/to/checkpoints

### 405B 模型训练配置（MoE）

In [ ]:
# 405B MoE 模型，512 GPU（64 节点 × 8 GPU）

torchrun \
    --nproc_per_node=8 \
    --nnodes=64 \
    ... \
    pretrain_gpt.py \
    --num-layers 126 \
    --hidden-size 16384 \
    --num-attention-heads 128 \
    --ffn-hidden-size 53248 \
    --num-experts 64 \                    # 64 个专家
    --expert-model-parallel-size 8 \       # EP=8
    --moe-router-topk 2 \                 # 每个 token 选 2 个专家
    --moe-token-dispatcher-type alltoall \
    \
    --tensor-model-parallel-size 8 \       # TP=8（同节点）
    --pipeline-model-parallel-size 8 \     # PP=8
    --context-parallel-size 2 \            # CP=2（长序列）
    --sequence-parallel \
    --use-distributed-optimizer \
    --data-parallel-sharding-strategy optim_grads_params \  # FSDP
    \
    --micro-batch-size 1 \
    --global-batch-size 2048 \
    ...

## 并行配置选择指南
### 决策流程

In [ ]:
模型大小？
│
├─ < 1B（单卡放得下）
│   └─ DP only，可能不需要 TP/PP
│
├─ 1B ~ 20B
│   └─ TP + DP
│       TP = 2~4（同节点）
│       DP = 总 GPU / TP
│
├─ 20B ~ 100B
│   └─ TP + PP + DP
│       TP = 4~8（同节点）
│       PP = 2~8（跨节点）
│       DP = 总 GPU / (TP × PP)
│
├─ 100B ~ 1T
│   └─ TP + PP + DP + SP
│       TP = 8（同节点满）
│       PP = 4~16
│       DP = 总 GPU / (TP × PP)
│       SP = TP（自动）
│
└─ > 1T 或 长序列
    └─ TP + PP + DP + SP + CP (+ EP for MoE)
        TP = 8
        PP = 8~32
        CP = 2~8（序列 > 64K）
        EP = 4~16（MoE）

### 常见配置速查表

| 模型大小     | GPU 数 | TP | PP | DP | CP  | 序列长度  | 显存/GPU |
| -------- | ----- | -- | -- | -- | --- | ----- | ------ |
| 7B       | 8     | 1  | 1  | 8  | 1   | 4096  | ~16GB  |
| 13B      | 16    | 2  | 1  | 8  | 1   | 4096  | ~20GB  |
| 30B      | 32    | 4  | 1  | 8  | 1   | 4096  | ~28GB  |
| 70B      | 64    | 4  | 4  | 4  | 1   | 4096  | ~40GB  |
| 175B     | 256   | 8  | 8  | 4  | 1   | 2048  | ~48GB  |
| 405B     | 512   | 8  | 8  | 8  | 1   | 4096  | ~60GB  |
| 405B MoE | 512   | 8  | 8  | 4  | 1   | 4096  | ~55GB  |
| 1T+      | 1024+ | 8  | 16 | 8+ | 2~4 | 8192+ | ~70GB  |


## 关键配置参数汇总

| 参数                                  | 说明          | 默认值       | 建议                                      |
| ----------------------------------- | ----------- | --------- | --------------------------------------- |
| `--tensor-model-parallel-size`      | 张量并行度       | 1         | 同节点内，≤8                                 |
| `--pipeline-model-parallel-size`    | 流水线并行度      | 1         | 跨节点                                     |
| `--context-parallel-size`           | 上下文并行度      | 1         | 长序列时 >1                                 |
| `--sequence-parallel`               | 启用序列并行      | False     | TP>1 时启用                                |
| `--expert-model-parallel-size`      | 专家并行度       | 1         | MoE 模型                                  |
| `--num-experts`                     | 专家数量        | 1         | MoE 模型                                  |
| `--use-distributed-optimizer`       | 分布式优化器      | False     | 大模型必开                                   |
| `--data-parallel-sharding-strategy` | ZeRO 级别     | no\_shard | optim/optim\_grads/optim\_grads\_params |
| `--micro-batch-size`                | 每 GPU batch | 1         | 显存允许尽量大                                 |
| `--global-batch-size`               | 总 batch     | 32        | DP × micro\_batch × grad\_accum         |
| `--pipeline-schedule`               | 流水线调度       | 1f1b      | 1f1b/interleaved/gpipe                  |


## 总结

| 并行维度   | 拆分对象                | 显存节省         | 通信开销          | 适用场景          |
| ------ | ------------------- | ------------ | ------------- | ------------- |
| **TP** | 层内参数（Attention/MLP） | 中            | 高（NVLink）     | 单卡放不下某层       |
| **PP** | 层间分配                | 高            | 低（P2P）        | 模型层数多         |
| **DP** | 数据分片                | 无（或 ZeRO 节省） | 中（All-Reduce） | 扩展 batch size |
| **SP** | 序列维度                | 中            | 高（NVLink）     | 长序列 + TP      |
| **CP** | 上下文分片               | 高            | 中（Ring）       | 超长序列（>64K）    |
| **EP** | MoE 专家              | 高            | 高（All-to-All） | MoE 模型        |

### 核心原则：
- TP 优先（同节点内，NVLink 带宽最高）
- PP 次之（跨节点，P2P 通信小）
- DP 兜底（扩展性最好）
- SP/CP 按需（长序列场景）
- EP MoE 专用（专家模型）